# First Solar Supply Chain - ML Model Training

This notebook trains and registers ML models for the First Solar Supply Chain Intelligence Agent:

1. **Demand Forecasting** — Gradient Boosting Regression for product-level module demand
2. **Stockout Risk Classification** — Predict materials at risk of breaching safety stock

Models are registered in the Snowflake Model Registry and called by SQL UDFs in `08_ml_model_functions.sql`.

In [ ]:
# Cell 1: Session setup
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
# Cell 2: Load demand forecast data for model training
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error, accuracy_score, classification_report

# Load product demand forecast (historical actuals for training)
df_forecast = session.sql("""
    SELECT PLANT_ID, PRODUCT_ID, WEEK_START, ACTUAL_MODULES, FORECAST_MODULES,
           LOWER_BOUND, UPPER_BOUND, IS_FUTURE
    FROM FS_INTELLIGENCE.RAW.PRODUCT_DEMAND_FORECAST
    WHERE IS_FUTURE = FALSE AND ACTUAL_MODULES IS NOT NULL
    ORDER BY PLANT_ID, PRODUCT_ID, WEEK_START
""").to_pandas()

print(f"Demand forecast training data: {len(df_forecast)} rows")
print(f"Plants: {df_forecast['PLANT_ID'].nunique()}, Products: {df_forecast['PRODUCT_ID'].nunique()}")

In [ ]:
# Cell 3: Feature engineering for demand forecast model
df_forecast['WEEK_START'] = pd.to_datetime(df_forecast['WEEK_START'])
df_forecast['WEEK_NUM'] = df_forecast.groupby(['PLANT_ID', 'PRODUCT_ID']).cumcount() + 1
df_forecast['MONTH'] = df_forecast['WEEK_START'].dt.month

# Encode categorical features
le_plant = LabelEncoder().fit(df_forecast['PLANT_ID'].unique())
le_prod = LabelEncoder().fit(df_forecast['PRODUCT_ID'].unique())
df_forecast['PLANT_ENC'] = le_plant.transform(df_forecast['PLANT_ID'])
df_forecast['PROD_ENC'] = le_prod.transform(df_forecast['PRODUCT_ID'])

# Per-series normalization
series_mean = df_forecast.groupby(['PLANT_ID', 'PRODUCT_ID'])['ACTUAL_MODULES'].transform('mean')
df_forecast['DEMAND_NORM'] = df_forecast['ACTUAL_MODULES'] / series_mean

# Lag features
df_forecast['LAG1'] = df_forecast.groupby(['PLANT_ID', 'PRODUCT_ID'])['DEMAND_NORM'].shift(1).fillna(1.0)
df_forecast['LAG2'] = df_forecast.groupby(['PLANT_ID', 'PRODUCT_ID'])['DEMAND_NORM'].shift(2).fillna(1.0)
df_forecast['ROLLING3'] = df_forecast.groupby(['PLANT_ID', 'PRODUCT_ID'])['DEMAND_NORM'].transform(
    lambda x: x.rolling(3, min_periods=1).mean().shift(1)
).fillna(1.0)

FEAT_COLS = ['WEEK_NUM', 'MONTH', 'LAG1', 'LAG2', 'ROLLING3', 'PLANT_ENC', 'PROD_ENC']
print(f"Features: {FEAT_COLS}")
print(f"Training samples: {len(df_forecast)}")

In [ ]:
# Cell 4: Train demand forecast model (Gradient Boosting Regression)
TRAIN_CUTOFF = int(len(df_forecast) * 0.8)
train_df = df_forecast.iloc[:TRAIN_CUTOFF]
val_df = df_forecast.iloc[TRAIN_CUTOFF:]

X_train = train_df[FEAT_COLS].values
y_train = train_df['DEMAND_NORM'].values
X_val = val_df[FEAT_COLS].values
y_val = val_df['DEMAND_NORM'].values

gbr_demand = GradientBoostingRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.08,
    subsample=0.8, min_samples_leaf=2, random_state=42
)
gbr_demand.fit(X_train, y_train)

# Evaluate
y_pred = gbr_demand.predict(X_val)
val_mape = mean_absolute_percentage_error(y_val, y_pred) * 100
print(f"Demand Forecast Model - Validation MAPE: {val_mape:.2f}%")
print(f"Feature importances: {dict(zip(FEAT_COLS, gbr_demand.feature_importances_.round(4)))}")

In [ ]:
# Cell 5: Register demand forecast model in Snowflake Model Registry
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name="FS_INTELLIGENCE", schema_name="ANALYTICS")

mv_demand = registry.log_model(
    gbr_demand,
    model_name="DEMAND_FORECAST_MODEL",
    version_name="V1",
    sample_input_data=pd.DataFrame(X_train[:5], columns=FEAT_COLS),
    conda_dependencies=["scikit-learn"],
    comment="GBR demand forecast model for First Solar module production. Predicts normalized weekly demand by plant/product."
)
mv_demand.set_metric("val_mape", val_mape)
mv_demand.set_metric("train_samples", len(X_train))
mv_demand.set_metric("val_samples", len(X_val))
print(f"Model registered: FS_INTELLIGENCE.ANALYTICS.DEMAND_FORECAST_MODEL (V1)")

In [ ]:
# Cell 6: Load inventory data for stockout risk model
df_inv = session.sql("""
    SELECT i.PLANT_ID, i.MATERIAL_ID, m.MATERIAL_CATEGORY, m.IS_CRITICAL,
           i.QUANTITY_ON_HAND, i.SAFETY_STOCK_LEVEL, i.DAYS_FORWARD_COVERAGE,
           i.REORDER_POINT, i.UNIT_COST,
           sm.LEAD_TIME_DAYS, sm.LEAD_TIME_VARIABILITY_DAYS,
           CASE WHEN i.QUANTITY_ON_HAND < i.SAFETY_STOCK_LEVEL THEN 1 ELSE 0 END AS IS_STOCKOUT_RISK
    FROM FS_INTELLIGENCE.RAW.INVENTORY_SNAPSHOT i
    JOIN FS_INTELLIGENCE.RAW.MATERIALS m ON i.MATERIAL_ID = m.MATERIAL_ID
    LEFT JOIN (
        SELECT PLANT_ID, MATERIAL_ID, MIN(LEAD_TIME_DAYS) AS LEAD_TIME_DAYS,
               MIN(LEAD_TIME_VARIABILITY_DAYS) AS LEAD_TIME_VARIABILITY_DAYS
        FROM FS_INTELLIGENCE.RAW.SUPPLIER_MATERIALS
        GROUP BY PLANT_ID, MATERIAL_ID
    ) sm ON i.PLANT_ID = sm.PLANT_ID AND i.MATERIAL_ID = sm.MATERIAL_ID
    ORDER BY i.SNAPSHOT_DATE DESC
""").to_pandas()

print(f"Stockout risk training data: {len(df_inv)} rows")
print(f"Stockout risk positive class: {df_inv['IS_STOCKOUT_RISK'].mean()*100:.1f}%")

In [ ]:
# Cell 7: Train and register stockout risk classifier
le_cat = LabelEncoder().fit(df_inv['MATERIAL_CATEGORY'].fillna('Unknown'))
df_inv['CAT_ENC'] = le_cat.transform(df_inv['MATERIAL_CATEGORY'].fillna('Unknown'))

RISK_FEATURES = ['QUANTITY_ON_HAND', 'SAFETY_STOCK_LEVEL', 'DAYS_FORWARD_COVERAGE',
                 'REORDER_POINT', 'UNIT_COST', 'LEAD_TIME_DAYS',
                 'LEAD_TIME_VARIABILITY_DAYS', 'CAT_ENC', 'IS_CRITICAL']
df_inv_clean = df_inv.dropna(subset=RISK_FEATURES + ['IS_STOCKOUT_RISK'])
df_inv_clean['IS_CRITICAL'] = df_inv_clean['IS_CRITICAL'].astype(int)

X_risk = df_inv_clean[RISK_FEATURES].values
y_risk = df_inv_clean['IS_STOCKOUT_RISK'].values

# Train/val split
split = int(len(X_risk) * 0.8)
X_tr_risk, X_va_risk = X_risk[:split], X_risk[split:]
y_tr_risk, y_va_risk = y_risk[:split], y_risk[split:]

gbc_stockout = GradientBoostingClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.1,
    subsample=0.8, random_state=42
)
gbc_stockout.fit(X_tr_risk, y_tr_risk)

acc = accuracy_score(y_va_risk, gbc_stockout.predict(X_va_risk))
print(f"Stockout Risk Classifier - Validation Accuracy: {acc:.3f}")
print(classification_report(y_va_risk, gbc_stockout.predict(X_va_risk), target_names=['Low Risk', 'High Risk']))

In [ ]:
# Cell 8: Register stockout risk model
mv_stockout = registry.log_model(
    gbc_stockout,
    model_name="STOCKOUT_RISK_MODEL",
    version_name="V1",
    sample_input_data=pd.DataFrame(X_tr_risk[:5], columns=RISK_FEATURES),
    conda_dependencies=["scikit-learn"],
    comment="GBC stockout risk classifier. Predicts whether a material will breach safety stock based on current inventory, lead times, and material properties."
)
mv_stockout.set_metric("accuracy", acc)
mv_stockout.set_metric("train_samples", len(X_tr_risk))
print(f"Model registered: FS_INTELLIGENCE.ANALYTICS.STOCKOUT_RISK_MODEL (V1)")
print(f"\nBoth models registered successfully. UDFs in 08_ml_model_functions.sql can now call them.")